# Voice of Customer Analysis with Snowflake

This notebook provides an interactive analysis workflow for voice-of-customer data using Snowflake's AI capabilities.

## Overview
The analysis follows a 5-step process:
1. **Select your Table/View/Stream** - Choose the data source
2. **Select your column** - Choose the text column to analyze
3. **Additional Configuration** - Set translation and grouping options
4. **Review** - Preview topics and output
5. **Save Results** - Persist results to a table

## Prerequisites
- Snowflake connection configured
- Required Python packages installed
- Access to Snowflake Cortex AI functions


## Step 0: Setup and Connection

First, let's import the necessary libraries and establish a connection to Snowflake.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from snowflake.snowpark import Session
from snowflake.snowpark import functions as F
from snowflake.snowpark import types as T
import warnings
warnings.filterwarnings('ignore')

# Set up plotting
plt.style.use('default')
sns.set_palette("husl")

print("📦 Libraries imported successfully!")


In [ ]:
# Establish Snowflake connection
try:
    session = Session.builder.getOrCreate()
    print("✅ Connected to Snowflake successfully!")
    
    # Test connection with a simple query
    version_result = session.sql("SELECT CURRENT_VERSION()").collect()
    print(f"🔗 Snowflake version: {version_result[0][0]}")
    
except Exception as e:
    print(f"❌ Failed to connect to Snowflake: {str(e)}")
    print("Please ensure your Snowflake connection is configured properly (environment variables, config file, etc.)")
    session = None


## Step 1: Select your Table/View/Stream

Let's explore the available databases, schemas, and tables in your Snowflake environment.


In [ ]:
if session is not None:
    # Get available databases
    databases = session.sql("SHOW TERSE DATABASES").collect()
    database_names = [db["name"] for db in databases]
    
    print("📊 Available Databases:")
    for i, db in enumerate(database_names, 1):
        print(f"  {i}. {db}")
    
    # Get available warehouses
    warehouses = session.sql("SHOW TERSE WAREHOUSES").collect()
    warehouse_names = [w["name"] for w in warehouses]
    
    print("\n🏭 Available Warehouses:")
    for i, wh in enumerate(warehouse_names, 1):
        print(f"  {i}. {wh}")
else:
    print("❌ No Snowflake connection available")


In [ ]:
# Configuration - Update these values based on your data
SELECTED_DATABASE = "YOUR_DATABASE"  # Replace with your database name
SELECTED_SCHEMA = "YOUR_SCHEMA"      # Replace with your schema name
SELECTED_TABLE = "YOUR_TABLE"        # Replace with your table name

print(f"🎯 Selected: {SELECTED_DATABASE}.{SELECTED_SCHEMA}.{SELECTED_TABLE}")
print("\n💡 Update the variables above with your actual database, schema, and table names")


## Step 2: Select your column

Now let's examine the structure of your selected table and choose the text column to analyze.


In [ ]:
if session is not None and SELECTED_DATABASE != "YOUR_DATABASE":
    try:
        # Get table structure
        table_name = f"{SELECTED_DATABASE}.{SELECTED_SCHEMA}.{SELECTED_TABLE}"
        df = session.table(table_name)
        
        # Show table info
        print(f"📊 Table: {table_name}")
        print(f"📏 Row count: {df.count()}")
        print(f"📋 Columns: {len(df.columns)}")
        
        # Show column information
        print("\n🔍 Column Details:")
        for i, col in enumerate(df.columns, 1):
            print(f"  {i}. {col}")
            
        # Show sample data
        print("\n📄 Sample Data (first 5 rows):")
        sample_df = df.limit(5).to_pandas()
        display(sample_df)
        
    except Exception as e:
        print(f"❌ Error accessing table: {str(e)}")
else:
    print("⚠️ Please update the database configuration above")


In [ ]:
# Configuration - Select the text column to analyze
SELECTED_COLUMN = "YOUR_TEXT_COLUMN"  # Replace with your text column name
GROUPING_COLUMN = None  # Optional: column to group topics by (e.g., "category", "product")
TRANSLATE_TO_ENGLISH = False  # Set to True if you want to translate non-English text

print(f"🎯 Selected text column: {SELECTED_COLUMN}")
print(f"📊 Grouping column: {GROUPING_COLUMN if GROUPING_COLUMN else 'None (analyze entire dataset)'}")
print(f"🌐 Translate to English: {TRANSLATE_TO_ENGLISH}")
print("\n💡 Update the variables above with your actual column names")


## Step 3: Additional Configuration

Let's set up the analysis parameters and examine the data distribution.


In [ ]:
if session is not None and SELECTED_COLUMN != "YOUR_TEXT_COLUMN":
    try:
        table_name = f"{SELECTED_DATABASE}.{SELECTED_SCHEMA}.{SELECTED_TABLE}"
        df = session.table(table_name)
        
        # Analyze text column
        print(f"📊 Analyzing column: {SELECTED_COLUMN}")
        
        # Get basic statistics
        total_rows = df.count()
        non_null_count = df.filter(F.col(SELECTED_COLUMN).is_not_null()).count()
        null_count = total_rows - non_null_count
        
        print(f"\n📈 Data Quality:")
        print(f"  Total rows: {total_rows:,}")
        print(f"  Non-null values: {non_null_count:,} ({non_null_count/total_rows*100:.1f}%)")
        print(f"  Null values: {null_count:,} ({null_count/total_rows*100:.1f}%)")
        
        # Get text length statistics
        text_length_stats = df.select(
            F.length(F.col(SELECTED_COLUMN)).alias("text_length")
        ).filter(F.col(SELECTED_COLUMN).is_not_null()).to_pandas()
        
        if not text_length_stats.empty:
            print(f"\n📏 Text Length Statistics:")
            print(f"  Mean length: {text_length_stats['text_length'].mean():.0f} characters")
            print(f"  Median length: {text_length_stats['text_length'].median():.0f} characters")
            print(f"  Min length: {text_length_stats['text_length'].min()} characters")
            print(f"  Max length: {text_length_stats['text_length'].max()} characters")
            
            # Plot text length distribution
            plt.figure(figsize=(10, 6))
            plt.hist(text_length_stats['text_length'], bins=50, alpha=0.7, edgecolor='black')
            plt.xlabel('Text Length (characters)')
            plt.ylabel('Frequency')
            plt.title(f'Distribution of Text Length in {SELECTED_COLUMN}')
            plt.grid(True, alpha=0.3)
            plt.show()
        
        # Show sample text
        print(f"\n📄 Sample Text from {SELECTED_COLUMN}:")
        sample_texts = df.filter(F.col(SELECTED_COLUMN).is_not_null()).limit(3).to_pandas()
        for i, row in sample_texts.iterrows():
            text = str(row[SELECTED_COLUMN])
            preview = text[:200] + "..." if len(text) > 200 else text
            print(f"\n  Sample {i+1}: {preview}")
            
    except Exception as e:
        print(f"❌ Error analyzing column: {str(e)}")
else:
    print("⚠️ Please update the column configuration above")


## Step 4: Generate Topics

Now let's extract topics from the text data using Snowflake's AI capabilities.


In [ ]:
def extract_topics(df, column, grouping_column=None):
    """Extract topics from text data using Snowflake Cortex AI."""
    
    if grouping_column:
        # Group by specified column and extract topics
        topics = (
            df.group_by(grouping_column)
            .agg(F.array_agg(F.col(column), is_distinct=True).alias(f"{column}_ARRAY"))
            .with_column(
                "TOP_TOPICS",
                F.call_builtin(
                    "SNOWFLAKE.CORTEX.COMPLETE",
                    F.lit("mistral-large2"),
                    F.concat(
                        F.lit(
                            "[SYSTEM] You are assigned to review various conversations and text snippets. Your role is to describe the issue. For example, Gloves missing price tags. Helmets were delivered with scratches."
                        ),
                        F.col(f"{column}_ARRAY").cast(T.StringType()),
                        F.lit(
                            "[USER] What are the main topics discussed by the customers? Do not respond with any preamble. Do not include 'Based on the conversations'. Do not include the name of the product in the topic."
                        ),
                    ),
                ),
            )
            .drop(f"{column}_ARRAY")
            .with_column("TOPICS_SPLIT", F.split(F.col("TOP_TOPICS"), F.lit("\n")))
            .join_table_function("FLATTEN", "TOPICS_SPLIT")
            .select(
                "*",
                F.regexp_replace(F.trim(F.col("VALUE")), r"^[-\d.]+\s*", "").alias("TOPIC"),
            )
            .group_by(grouping_column)
            .agg(F.array_agg(F.col("TOPIC")).alias("TOPIC_ARRAY"))
        )
    else:
        # Extract topics from entire dataset
        topics = (
            df.with_column(
                f"{column}_PARSED",
                F.parse_json(
                    F.call_builtin(
                        "SNOWFLAKE.CORTEX.COMPLETE",
                        "mistral-large2",
                        F.concat(
                            F.lit(
                                "Extract 5 topics from this field in an array format."
                            ),
                            F.col(column),
                            F.lit(
                                "Do not include any preambles. Do not include any backticks."
                            ),
                        ),
                    ),
                ),
            )
            .join_table_function("FLATTEN", f"{column}_PARSED")
            .select(F.col("VALUE").cast(T.StringType()).alias("VALUE"))
            .agg(F.array_agg(F.col("VALUE"), is_distinct=True).alias("TOPICS"))
            .with_column(
                "TOPIC_ARRAY",
                F.parse_json(
                    F.call_builtin(
                        F.lit("mistral-large2"),
                        F.concat(
                            F.lit("Given this list of topics: "),
                            F.col("TOPICS").cast(T.StringType()),
                            F.lit(
                                "- Combine any that are similar only returning the primary topic. Do not include any preamble. This should be a list of only ten. This should be in an array format."
                            ),
                        ),
                    ),
                ),
            )
        ).drop("TOPICS")
    
    return topics

print("🔧 Topic extraction function defined")


In [ ]:
if session is not None and SELECTED_COLUMN != "YOUR_TEXT_COLUMN":
    try:
        print("🔄 Extracting topics... This may take a few minutes.")
        
        table_name = f"{SELECTED_DATABASE}.{SELECTED_SCHEMA}.{SELECTED_TABLE}"
        df = session.table(table_name).filter(F.col(SELECTED_COLUMN).is_not_null())
        
        # Extract topics
        topics_df = extract_topics(df, SELECTED_COLUMN, GROUPING_COLUMN)
        
        print("✅ Topics extracted successfully!")
        print(f"📊 Topics DataFrame shape: {topics_df.count()} rows")
        
        # Display topics
        topics_pandas = topics_df.to_pandas()
        print("\n📋 Extracted Topics:")
        display(topics_pandas)
        
    except Exception as e:
        print(f"❌ Error extracting topics: {str(e)}")
        print("💡 Make sure you have access to Snowflake Cortex AI functions")
else:
    print("⚠️ Please update the column configuration above")


## Step 5: Analyze Sentiment and Generate Insights

Let's analyze sentiment and create comprehensive insights from the data.


In [ ]:
def translate_column(df, column):
    """Translate non-English text to English if needed."""
    return (
        df.with_column(
            "LANGUAGE",
            F.call_udf("voice_of_the_customer.app.detect", F.col(column)),
        )
        .with_column(
            f"TRANSLATED_{column}",
            F.when(F.col("LANGUAGE") == F.lit("en"), F.col(column)).otherwise(
                F.call_builtin(
                    "SNOWFLAKE.CORTEX.TRANSLATE",
                    F.col(column),
                    F.col("LANGUAGE"),
                    F.lit("en"),
                )
            ),
        )
        .drop(column)
        .with_column_renamed(f"TRANSLATED_{column}", column)
    )

def summarize_column(df, column):
    """Generate summaries for text data."""
    return df.with_column(
        "TRANSCRIPT_SUMMARY",
        F.call_builtin(
            "SNOWFLAKE.CORTEX.SUMMARIZE",
            F.col(column),
        ),
    )

def extract_sentiment(df, column):
    """Extract sentiment from text data."""
    return df.with_column(
        "SENTIMENT",
        F.call_builtin("SNOWFLAKE.CORTEX.SENTIMENT", F.col(column)),
    )

print("🔧 Analysis functions defined")


In [ ]:
if session is not None and SELECTED_COLUMN != "YOUR_TEXT_COLUMN":
    try:
        print("🔄 Running comprehensive analysis...")
        
        table_name = f"{SELECTED_DATABASE}.{SELECTED_SCHEMA}.{SELECTED_TABLE}"
        df = session.table(table_name).filter(F.col(SELECTED_COLUMN).is_not_null())
        
        # Apply translation if needed
        if TRANSLATE_TO_ENGLISH:
            print("🌐 Translating text to English...")
            df = translate_column(df, SELECTED_COLUMN)
        
        # Extract sentiment
        print("😊 Analyzing sentiment...")
        df = extract_sentiment(df, SELECTED_COLUMN)
        
        # Generate summaries
        print("📝 Generating summaries...")
        df = summarize_column(df, SELECTED_COLUMN)
        
        # Get results
        result_df = df.limit(100).to_pandas()  # Limit for display
        
        print("✅ Analysis completed!")
        print(f"📊 Results shape: {result_df.shape}")
        
        # Display results
        print("\n📋 Analysis Results:")
        display(result_df.head())
        
        # Sentiment distribution
        if 'SENTIMENT' in result_df.columns:
            sentiment_counts = result_df['SENTIMENT'].value_counts()
            print("\n😊 Sentiment Distribution:")
            print(sentiment_counts)
            
            # Plot sentiment distribution
            plt.figure(figsize=(10, 6))
            sentiment_counts.plot(kind='bar', color=['#ff6b6b', '#4ecdc4', '#45b7d1'])
            plt.title('Sentiment Distribution')
            plt.xlabel('Sentiment')
            plt.ylabel('Count')
            plt.xticks(rotation=45)
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()
        
    except Exception as e:
        print(f"❌ Error in analysis: {str(e)}")
        print("💡 Make sure you have access to Snowflake Cortex AI functions")
else:
    print("⚠️ Please update the column configuration above")


## Step 6: Save Results (Optional)

Save the analysis results to a new table in Snowflake.


In [ ]:
# Configuration for saving results
SAVE_RESULTS = False  # Set to True to save results
OUTPUT_DATABASE = "YOUR_DATABASE"    # Replace with your output database
OUTPUT_SCHEMA = "YOUR_SCHEMA"        # Replace with your output schema
OUTPUT_TABLE = "voc_analysis_results"  # Name for the output table

print(f"💾 Save results: {SAVE_RESULTS}")
print(f"📁 Output location: {OUTPUT_DATABASE}.{OUTPUT_SCHEMA}.{OUTPUT_TABLE}")
print("\n💡 Update the configuration above to save results")


In [ ]:
if SAVE_RESULTS and session is not None and SELECTED_COLUMN != "YOUR_TEXT_COLUMN":
    try:
        print("💾 Saving results to Snowflake...")
        
        # Recreate the full analysis
        table_name = f"{SELECTED_DATABASE}.{SELECTED_SCHEMA}.{SELECTED_TABLE}"
        df = session.table(table_name).filter(F.col(SELECTED_COLUMN).is_not_null())
        
        # Apply all transformations
        if TRANSLATE_TO_ENGLISH:
            df = translate_column(df, SELECTED_COLUMN)
        
        df = extract_sentiment(df, SELECTED_COLUMN)
        df = summarize_column(df, SELECTED_COLUMN)
        
        # Save to new table
        output_table_name = f"{OUTPUT_DATABASE}.{OUTPUT_SCHEMA}.{OUTPUT_TABLE}"
        df.write.save_as_table(
            output_table_name,
            mode="overwrite"
        )
        
        print(f"✅ Results saved to {output_table_name}")
        print(f"📊 Total rows saved: {df.count()}")
        
    except Exception as e:
        print(f"❌ Error saving results: {str(e)}")
else:
    print("⚠️ Results not saved (SAVE_RESULTS is False or configuration incomplete)")


## Summary and Next Steps

### What we accomplished:
1. ✅ Connected to Snowflake
2. ✅ Explored available databases and tables
3. ✅ Analyzed text data quality and distribution
4. ✅ Extracted topics using AI
5. ✅ Analyzed sentiment
6. ✅ Generated summaries
7. ✅ (Optional) Saved results to a table

### Next steps you can take:
- **Customize the analysis**: Modify the prompts and parameters
- **Add more visualizations**: Create charts for topic distribution, sentiment trends
- **Export results**: Download results as CSV or other formats
- **Schedule analysis**: Set up automated runs using Snowflake tasks
- **Integrate with other tools**: Connect to BI tools or other analytics platforms

### Tips for better results:
- Ensure your text data is clean and well-formatted
- Consider grouping by relevant dimensions (product, time, customer segment)
- Experiment with different AI models and prompts
- Monitor costs as AI functions can be expensive for large datasets

Happy analyzing! 🎉
